In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pickle
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
graph_embeddings = torch.load(
    "/content/drive/MyDrive/ds003844_processed/spatial_graph_embeddings.pt",
    map_location=device
)
print(graph_embeddings.shape)

torch.Size([3684, 128])


In [ ]:
with open(
    "/content/drive/MyDrive/ds003844_processed/recording_metadata.pkl",
    "rb"
) as f:
    metadata = pickle.load(f)
sequence_lengths = metadata["sequence_lengths"]
recording_names = metadata["recording_names"]
channel_counts = metadata["channel_counts"]

In [ ]:
sequences = []
idx = 0
for length in sequence_lengths:
    sequences.append(
        graph_embeddings[idx:idx + length]
    )
    idx += length

In [ ]:
print("Number of sequences:", len(sequences))
for i in range(5):
    print(f"Recording {i+1}: {sequences[i].shape}")
reconstructed = torch.cat(sequences, dim=0)
print("\nReconstruction Correct:",torch.allclose(reconstructed, graph_embeddings))

Number of sequences: 33
Recording 1: torch.Size([128, 128])
Recording 2: torch.Size([123, 128])
Recording 3: torch.Size([154, 128])
Recording 4: torch.Size([310, 128])
Recording 5: torch.Size([236, 128])

Reconstruction Correct: True


In [ ]:
from torch.utils.data import Dataset
class BrainSequenceDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        return self.sequences[idx]

In [ ]:
dataset = BrainSequenceDataset(sequences)
print(len(dataset))

33


In [ ]:
max_len = max(sequence_lengths)
print("Maximum sequence length:", max_len)

Maximum sequence length: 310


In [ ]:
from torch.nn.utils.rnn import pad_sequence
padded_sequences = pad_sequence(
    sequences,
    batch_first=True,
    padding_value=0.0
)
print(padded_sequences.shape)

torch.Size([33, 310, 128])


In [ ]:
attention_mask = torch.zeros(
    len(sequence_lengths),
    max_len,
    dtype=torch.bool
)
for i, length in enumerate(sequence_lengths):
    attention_mask[i, :length] = True

In [ ]:
print(attention_mask.shape)
print(attention_mask[0])

torch.Size([33, 310])
tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,

In [ ]:
import math
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
positional_encoding = PositionalEncoding(
    d_model=128,
    max_len=max_len
).to(device)
x = positional_encoding(padded_sequences)
print(x.shape)

torch.Size([33, 310, 128])


In [ ]:
encoder_layer = nn.TransformerEncoderLayer(
    d_model=128,
    nhead=8,
    dim_feedforward=512,
    dropout=0.1,
    batch_first=True
)

transformer = nn.TransformerEncoder(
    encoder_layer,
    num_layers=2
).to(device)

In [ ]:
src_padding_mask = (~attention_mask).to(device)
x = positional_encoding(padded_sequences)
embeddings = transformer(
    x,
    src_key_padding_mask=src_padding_mask
)
print(embeddings.shape)

torch.Size([33, 310, 128])


In [ ]:
reconstruction_head = nn.Linear(
    128,
    128
).to(device)

In [ ]:
reconstructed = reconstruction_head(embeddings)
print(reconstructed.shape)

torch.Size([33, 310, 128])


In [ ]:
attention_mask = attention_mask.to(device)
mask_ratio = 0.15
random_mask = (
    torch.rand(
        padded_sequences.shape[:2],
        device=device
    ) < mask_ratio
)
# Never mask padding positions
random_mask &= attention_mask
print(random_mask.shape)
print(random_mask.sum())

torch.Size([33, 310])
tensor(515, device='cuda:0')


In [ ]:
masked_input = padded_sequences.clone()
masked_input[random_mask] = 0.0

In [ ]:
idx = random_mask.nonzero()[0]
print("Masked position:", idx)
print("\nOriginal embedding:")
print(padded_sequences[idx[0], idx[1], :5])
print("\nMasked embedding:")
print(masked_input[idx[0], idx[1], :5])

Masked position: tensor([0, 6], device='cuda:0')

Original embedding:
tensor([ 0.3596,  0.1450, -0.0230,  0.5471, -0.1978], device='cuda:0')

Masked embedding:
tensor([0., 0., 0., 0., 0.], device='cuda:0')


In [ ]:
x = positional_encoding(masked_input)
embeddings = transformer(
    x,
    src_key_padding_mask=~attention_mask
)
reconstructed = reconstruction_head(embeddings)
print(reconstructed.shape)

torch.Size([33, 310, 128])


In [ ]:
criterion = nn.MSELoss()
loss = criterion(
    reconstructed[random_mask],
    padded_sequences[random_mask]
)
print(loss)

tensor(0.4100, device='cuda:0', grad_fn=<MseLossBackward0>)


In [ ]:
optimizer = torch.optim.Adam(
    list(transformer.parameters()) +
    list(reconstruction_head.parameters()),
    lr=1e-4
)

In [ ]:
epochs = 1000
for epoch in range(epochs):
    transformer.train()
    reconstruction_head.train()
    # Random masking every epoch
    random_mask = (
        torch.rand(
            padded_sequences.shape[:2],
            device=device
        ) < 0.15
    )
    random_mask &= attention_mask
    masked_input = padded_sequences.clone()
    masked_input[random_mask] = 0.0
    optimizer.zero_grad()
    x = positional_encoding(masked_input)
    embeddings = transformer(
        x,
        src_key_padding_mask=~attention_mask
    )
    reconstructed = reconstruction_head(embeddings)
    loss = criterion(
        reconstructed[random_mask],
        padded_sequences[random_mask]
    )
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss: {loss.item():.6f}"
        )

Epoch 10/1000 | Loss: 0.225585
Epoch 20/1000 | Loss: 0.117265
Epoch 30/1000 | Loss: 0.060282
Epoch 40/1000 | Loss: 0.034599
Epoch 50/1000 | Loss: 0.025862
Epoch 60/1000 | Loss: 0.021236
Epoch 70/1000 | Loss: 0.019271
Epoch 80/1000 | Loss: 0.018013
Epoch 90/1000 | Loss: 0.017206
Epoch 100/1000 | Loss: 0.016388
Epoch 110/1000 | Loss: 0.015398
Epoch 120/1000 | Loss: 0.015002
Epoch 130/1000 | Loss: 0.014980
Epoch 140/1000 | Loss: 0.014128
Epoch 150/1000 | Loss: 0.013540
Epoch 160/1000 | Loss: 0.013592
Epoch 170/1000 | Loss: 0.013161
Epoch 180/1000 | Loss: 0.012994
Epoch 190/1000 | Loss: 0.012620
Epoch 200/1000 | Loss: 0.012068
Epoch 210/1000 | Loss: 0.011984
Epoch 220/1000 | Loss: 0.011887
Epoch 230/1000 | Loss: 0.011345
Epoch 240/1000 | Loss: 0.010971
Epoch 250/1000 | Loss: 0.010840
Epoch 260/1000 | Loss: 0.010775
Epoch 270/1000 | Loss: 0.010569
Epoch 280/1000 | Loss: 0.010223
Epoch 290/1000 | Loss: 0.009922
Epoch 300/1000 | Loss: 0.009870
Epoch 310/1000 | Loss: 0.009683
Epoch 320/1000 | 

In [ ]:
transformer.eval()
with torch.no_grad():
    x = positional_encoding(padded_sequences)
    spatio_temporal_embeddings = transformer(
        x,
        src_key_padding_mask=~attention_mask
    )

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


In [ ]:
final_embeddings = []
for i, length in enumerate(sequence_lengths):
    final_embeddings.append(
        spatio_temporal_embeddings[i, :length]
    )
final_embeddings = torch.cat(
    final_embeddings,
    dim=0
)
print(final_embeddings.shape)

torch.Size([3684, 128])


In [ ]:
torch.save(
    final_embeddings.cpu(),
    "/content/drive/MyDrive/ds003844_processed/spatio_temporal_embeddings.pt"
)

In [ ]:
torch.save(
    transformer.state_dict(),
    "/content/drive/MyDrive/ds003844_processed/masked_transformer.pt"
)

In [ ]:
metadata = {
    "sequence_lengths": sequence_lengths,
    "recording_names": recording_names,
    "channel_counts": channel_counts
}
with open(
    "/content/drive/MyDrive/ds003844_processed/spatio_temporal_metadata.pkl",
    "wb"
) as f:
    pickle.dump(metadata, f)